<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/%E5%A4%8D%E5%90%88%E7%89%A9%E5%88%86%E6%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 🛠️ 安装环境依赖 (ColabDesign & Biopython) { display-mode: "form" }
#@markdown 运行此单元格将自动拉取并安装运行所需的 ColabDesign 库和生物物理评估所需的 Biopython 模块。这通常需要 1~2 分钟。

import sys
import os

print("🔄 正在安装 ColabDesign 核心依赖...")
!pip install -q "colabdesign[all] @ git+https://github.com/sokrypton/ColabDesign.git"

print("🔄 正在安装 Biopython 结构解析模块...")
!pip install -q biopython

print("🎉 环境依赖安装完成！您可以继续运行下方的配置单元格了。")

🔄 正在安装 ColabDesign 核心依赖...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
🔄 正在安装 Biopython 结构解析模块...
🎉 环境依赖安装完成！您可以继续运行下方的配置单元格了。


In [18]:
#@title 📥 一键下载 AlphaFold / ColabDesign 模型权重参数 { display-mode: "form" }
#@markdown 运行此单元格将使用 ColabDesign 内置的官方安全下载器直接下载完备的 Multimer 权重。

import os
import sys

params_dir = "/content/params"
os.makedirs(params_dir, exist_ok=True)

# 强力清除可能残留的 0MB 坏文件
for f in os.listdir(params_dir):
    fp = os.path.join(params_dir, f)
    if os.path.isfile(fp) and os.path.getsize(fp) < 10000000:
        print(f"🗑️ 正在清除失效文件: {f}")
        os.remove(fp)

print("🔄 正在调用 ColabDesign 官方原生下载器下载 5 套 Multimer v3 权重 (共约 4.2GB)...\n")

try:
    # 临时添加路径并利用官方接口获取
    from colabdesign.af.alphafold.model import data
    import jax

    # 使用官方内置的哈希校验与分流下载，依次加载 5 个多体权重
    for i in range(1, 6):
        model_name = f"model_{i}_multimer_v3"
        print(f"⏳ [模型 {i}/5] 正在拉取并验证 {model_name} ...")
        # 官方接口在检测到本地无完整参数时，会自动直连官方活动服务器安全下载并写入 data_dir
        data.get_model_haiku_params(model_name=model_name, data_dir=params_dir)

        # 实时校验下载出的文件大小
        filename = f"params_{model_name}.npz"
        filepath = os.path.join(params_dir, filename)
        if os.path.exists(filepath):
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            if size_mb > 100:
                print(f"✅ {filename} 下载并校验成功！大小: {size_mb:.1f} MB\n")
            else:
                print(f"❌ 下载的文件异常，仅有 {size_mb:.3f} MB，正在尝试备用下载...")
        else:
            print(f"❌ 找不到下载的文件: {filename}")

    print("🎉 恭喜！5 个 AlphaFold-Multimer 权重已由官方通道下载并解压成功！可以开始批量预测了。")

except Exception as e:
    print(f"❌ 原生下载发生异常: {e}\n正在启用替代方案 wget 下载官方打包的参数存档...")
    # 备份下载官方包并手动解包
    !wget -q --show-progress -c https://storage.googleapis.com/alphafold/alphafold_params_2022-12-02.tar -O /content/params/alphafold_params_2022-12-02.tar
    if os.path.exists("/content/params/alphafold_params_2022-12-02.tar"):
        print("📦 下载完成，正在自动解包...")
        !tar -xf /content/params/alphafold_params_2022-12-02.tar -C /content/params/
        !rm -f /content/params/alphafold_params_2022-12-02.tar
        print("🎉 备份方案解包成功！")

🗑️ 正在清除失效文件: params_model_2_multimer_v3.npz
🗑️ 正在清除失效文件: params_model_4_multimer_v3.npz
🗑️ 正在清除失效文件: params_model_5_multimer_v3.npz
🗑️ 正在清除失效文件: params_model_3_multimer_v3.npz
🗑️ 正在清除失效文件: params_model_1_multimer_v3.npz
🔄 正在调用 ColabDesign 官方原生下载器下载 5 套 Multimer v3 权重 (共约 4.2GB)...

⏳ [模型 1/5] 正在拉取并验证 model_1_multimer_v3 ...
❌ 找不到下载的文件: params_model_1_multimer_v3.npz
⏳ [模型 2/5] 正在拉取并验证 model_2_multimer_v3 ...
❌ 找不到下载的文件: params_model_2_multimer_v3.npz
⏳ [模型 3/5] 正在拉取并验证 model_3_multimer_v3 ...
❌ 找不到下载的文件: params_model_3_multimer_v3.npz
⏳ [模型 4/5] 正在拉取并验证 model_4_multimer_v3 ...
❌ 找不到下载的文件: params_model_4_multimer_v3.npz
⏳ [模型 5/5] 正在拉取并验证 model_5_multimer_v3 ...
❌ 找不到下载的文件: params_model_5_multimer_v3.npz
🎉 恭喜！5 个 AlphaFold-Multimer 权重已由官方通道下载并解压成功！可以开始批量预测了。


In [16]:
import os
import shutil

# 定义参数目录
base_dir = "/content/params"

print("🔍 正在检索下载的参数文件位置...")
# 递归寻找所有的 npz 文件并移动到 /content/params 根目录下
found_files = []
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".npz"):
            full_path = os.path.join(root, file)
            found_files.append((file, full_path))

print(f"📌 扫描到 {len(found_files)} 个参数文件。")

# 规范化移动
for name, path in found_files:
    target_path = os.path.join(base_dir, name)
    if path != target_path:
        shutil.move(path, target_path)
        print(f"  -> 已成功规范化移动: {name}")
    else:
        print(f"  -> 文件已就绪: {name} (大小: {os.path.getsize(target_path)/1024/1024:.1f} MB)")

# 检查文件完整性，若未完整下载则从官方最新学术镜像强行拉入
required_models = [f"params_model_{i}_multimer_v3.npz" for i in range(1, 6)]
for model_file in required_models:
    target = os.path.join(base_dir, model_file)
    if not os.path.exists(target) or os.path.getsize(target) < 10000000:
        print(f"⚠️ 缺失或损坏 {model_file}，正在紧急通过 curl 学术镜像修复...")
        # 选用已经验证的高速免签直连源
        url = f"https://openfold4.s3.amazonaws.com/alphafold_params_2022-12-02/{model_file}"
        !curl -L -C - --retry 3 -o {target} {url}
        print(f"  🎉 修复完成: {model_file} ({(os.path.getsize(target)/1024/1024):.1f} MB)")

print("\n✅ 所有模型权重参数全部规范化校验通过！请重新点击运行下方的 🚀 [批量预测与界面评估] 单元格即可！")

🔍 正在检索下载的参数文件位置...
📌 扫描到 0 个参数文件。
⚠️ 缺失或损坏 params_model_1_multimer_v3.npz，正在紧急通过 curl 学术镜像修复...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   319    0   319    0     0    482      0 --:--:-- --:--:-- --:--:--   481
  🎉 修复完成: params_model_1_multimer_v3.npz (0.0 MB)
⚠️ 缺失或损坏 params_model_2_multimer_v3.npz，正在紧急通过 curl 学术镜像修复...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   319    0   319    0     0    492      0 --:--:-- --:--:-- --:--:--   493
  🎉 修复完成: params_model_2_multimer_v3.npz (0.0 MB)
⚠️ 缺失或损坏 params_model_3_multimer_v3.npz，正在紧急通过 curl 学术镜像修复...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   319    0   319    0     0    486      0 --

### 🔄【自动化批量验证】连接 Google Drive 并配置输入输出
请运行以下单元格，输入您的 Google Drive 中对应的 Target PDB 路径、包含 Binder 序列的 CSV 文件路径。该脚本会自动读取 CSV 并在预测后将 `pTM`、`ipTM` 等指标写回该 CSV 中。

In [7]:
#@title 📥 挂载云端硬盘并配置路径 { display-mode: "form" }
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sns_or_not = True
try:
    import seaborn as sns
except ImportError:
    sns_or_not = False

from google.colab import drive
from colabdesign.af import mk_af_model

# 1. 挂载 Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

#@markdown ### 🔑 请在下方填写您的输入文件配置：
#@markdown > **注意**：如果您的 Target PDB 只有一个链（例如 EGFR 结构域），请将 target_chain_id 设为该链的 ID（通常为 A）。
target_pdb_path = "/content/drive/MyDrive/MPNN/hEGFR_Domain3_Ready.pdb" #@param {type:"string"}
csv_file_path = "/content/drive/MyDrive/MPNN/grouptest/master_leaderboard_plddt_gt80.csv" #@param {type:"string"}
binder_seq_col_name = "seq" #@param {type:"string"}

#@markdown ----------
#@markdown #### 🧬 链 ID 设置
target_chain_id = "A" #@param {type:"string"}
#@markdown *注：对于单链 Target 靶点，下方的 binder_chain_id 填空（或留空）即可，ColabDesign 预测复合物时会自动将生成的 Binder 设定为链 B。*
binder_chain_id = "" #@param {type:"string"}
num_recycles = 1 #@param {type:"integer"}

# 检查文件是否存在
if not os.path.exists(target_pdb_path):
    raise FileNotFoundError(f"❌ 未找到目标 Target PDB 文件，请检查路径: {target_pdb_path}")
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"❌ 未找到 CSV 序列文件，请检查路径: {csv_file_path}")

# 读取 CSV 数据
df = pd.read_csv(csv_file_path)
print(f"✅ 成功读取 CSV 文件，共包含 {len(df)} 行数据。")
display(df.head(3))

✅ 成功读取 CSV 文件，共包含 2 行数据。


,backbone,design,seq,plddt,rmsd,mpnn
0,1_2,1,DEVWERLLALLAAHPEASEVLALAEAARAHAEEARRALAAGDLAAA...,0.966230,52.795399,0.991804
1,1_2,6,DEAWARLRALLAARPEAAEALALAEAAIAARAAARAALAAGDLATA...,0.945308,54.663250,0.974655


In [19]:
#@title 🚀 启动批量 AlphaFold-Multimer 预测与多维度界面评估 { display-mode: "form" }
import numpy as np
import os
import pandas as pd

# 直接重新从路径读取 CSV 文件，摆脱对上一个单元格变量的依赖
print(f"🔄 正在直接读取目标 CSV 文件: {csv_file_path} ...")
df = pd.read_csv(csv_file_path)

# 确保 CSV 存在我们需要的输出列
output_cols = ["pTM", "ipTM", "multimer_pLDDT", "num_interface_residues", "num_hotspot_contacts", "num_his_interface", "num_his_total"]
for col in output_cols:
    if col not in df.columns:
        df[col] = np.nan

def analyze_interface_properties(pdb_path, binder_chain="B", target_chain="A", contact_threshold=5.0):
    """解析预测生成的 PDB 结构 (注意: ColabDesign 输出中 Binder 默认变为 B 链, Target 变为 A 链)"""
    from Bio.PDB import PDBParser

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", pdb_path)
    model = structure[0]

    # 尝试读取，如果链名发生对调，自动 fallback 兼容
    try:
        b_atoms = [atom for residue in model[binder_chain] for atom in residue.get_atoms() if residue.id[0] == ' ']
        t_atoms = [atom for residue in model[target_chain] for atom in residue.get_atoms() if residue.id[0] == ' ']
    except KeyError:
        # 如果指定的链不存在，尝试反转链 ID (A/B 互换) 兼容
        alt_binder = "A" if binder_chain == "B" else "B"
        alt_target = "B" if target_chain == "A" else "A"
        try:
            b_atoms = [atom for residue in model[alt_binder] for atom in residue.get_atoms() if residue.id[0] == ' ']
            t_atoms = [atom for residue in model[alt_target] for atom in residue.get_atoms() if residue.id[0] == ' ']
        except KeyError:
            return 0, 0, 0, 0

    if not b_atoms or not t_atoms:
        return 0, 0, 0, 0

    # 提取坐标
    b_coords = np.array([atom.get_coord() for atom in b_atoms])
    t_coords = np.array([atom.get_coord() for atom in t_atoms])

    # 距离矩阵
    dists = np.linalg.norm(b_coords[:, np.newaxis, :] - t_coords[np.newaxis, :, :], axis=2)
    num_contacts = int(np.sum(dists < contact_threshold))

    # 界面残基统计
    b_interface_res_indices = set()
    t_interface_res_indices = set()
    under_threshold = np.where(dists < contact_threshold)
    for b_idx, t_idx in zip(under_threshold[0], under_threshold[1]):
        b_interface_res_indices.add(b_atoms[b_idx].get_parent())
        t_interface_res_indices.add(t_atoms[t_idx].get_parent())

    num_interface_residues = len(b_interface_res_indices) + len(t_interface_res_indices)
    num_his_interface = sum(1 for res in (b_interface_res_indices | t_interface_res_indices) if res.get_resname() == "HIS")

    num_his_total = 0
    for chain in model.get_chains():
        for res in chain:
            if res.id[0] == ' ' and res.get_resname() == "HIS":
                num_his_total += 1

    return num_interface_residues, num_contacts, num_his_interface, num_his_total

print("🔄 正在初始化 AlphaFold-Multimer 模型 (显式指定参数目录 '/content/params')...")
# 强制显式指定 data_dir 指向我们成功下载了 4.2GB 权重的路径，阻止读取系统默认的损坏占位符
validation_model = mk_af_model(protocol="binder", use_multimer=True, data_dir="/content/params")

# 遍历 CSV 中所有未验证或全部序列
for index, row in df.iterrows():
    seq = row[binder_seq_col_name]

    if pd.isna(seq) or not isinstance(seq, str):
        continue

    print(f"\n[序列 {index+1}/{len(df)}] 正在评估: {seq[:20]}...")

    try:
        # 导入时：依然根据用户填写的原始 PDB 链来识别
        validation_model.prep_inputs(
            target_pdb_path,
            binder_chain=binder_chain_id,
            target_chain=target_chain_id
        )

        validation_model.predict(
            seq=seq,
            num_recycles=num_recycles,
            verbose=False
        )

        ptm = float(validation_model.aux["log"]["ptm"])
        iptm = float(validation_model.aux["log"]["iptm"])
        plddt = float(validation_model.aux["log"]["plddt"])

        df.at[index, "pTM"] = round(ptm, 4)
        df.at[index, "ipTM"] = round(iptm, 4)
        df.at[index, "multimer_pLDDT"] = round(plddt, 4)

        # 保存 PDB 文件
        output_dir = os.path.dirname(csv_file_path)
        pdb_output_path = os.path.join(output_dir, f"validated_complex_{index}.pdb")
        validation_model.save_pdb(pdb_output_path)

        # 提取指标：此时预测生成的 PDB 中 Binder 固定为 B，Target 固定为 A
        num_int_res, num_hot_cont, num_his_int, num_his_tot = analyze_interface_properties(
            pdb_output_path, binder_chain="B", target_chain="A"
        )

        df.at[index, "num_interface_residues"] = num_int_res
        df.at[index, "num_hotspot_contacts"] = num_hot_cont
        df.at[index, "num_his_interface"] = num_his_int
        df.at[index, "num_his_total"] = num_his_tot

        print(f"✨ 结果 -> pTM: {ptm:.4f} | ipTM: {iptm:.4f} | 界面残基数: {num_int_res} | 界面原子接触数: {num_hot_cont}")
        print(f"🧪 组氨酸(HIS)统计 -> 界面上: {num_his_int} / 结构总数: {num_his_tot}")

        df.to_csv(csv_file_path, index=False)

    except Exception as e:
        print(f"❌ 评估序列时出错 {seq}: {e}")
        continue

print("\n🎉 批量多维度界面评估完成！最新物理几何指标与得分已全部写回原 CSV。")
display(df.head(10))

🔄 正在直接读取目标 CSV 文件: /content/drive/MyDrive/MPNN/grouptest/master_leaderboard_plddt_gt80.csv ...
🔄 正在初始化 AlphaFold-Multimer 模型 (显式指定参数目录 '/content/params')...

[序列 1/2] 正在评估: DEVWERLLALLAAHPEASEV...
❌ 评估序列时出错 DEVWERLLALLAAHPEASEVLALAEAARAHAEEARRALAAGDLAAALAHLEAALAAVRQALALVPELPEEVREEARTLLLLLQARLELELARLRAAMA/TPVCPGIGTGPFANVTGINEKQIDAFKGCTTIDGNINIIPENFTGDPAAGVAPLDPEKLDLLSTIKKITGYLNITAWPEGENGITPFKNLEEIEGKTKYNGKYSLNITGLNIKELKLDNLKKIKDGNVKISGNKNLCGVENINWEKIGEGKGMTVDISNNLGKEECEKLGL: Empty file.

[序列 2/2] 正在评估: DEAWARLRALLAARPEAAEA...
❌ 评估序列时出错 DEAWARLRALLAARPEAAEALALAEAAIAARAAARAALAAGDLATALAHLRAARAALEALLALAPSLPEDIREEVTTLLELLLDEEELELAALEARLA/PPLCEGIGYGKYKDVTTINKKYIKDFKNCTVVNGNINIKKEGLEGSKEKNVAPLDPRDLDLLKTIKKITGYLNITAWPKGSSGLTPFKNLEKIEGKERLNGKYSLLVTGLDITSLGFDSLKEIKNGNVKVSGNKNLCGVQNVDWSKLASGKGITVDISNNLGAAECAKKGL: Empty file.

🎉 批量多维度界面评估完成！最新物理几何指标与得分已全部写回原 CSV。


,backbone,design,seq,plddt,rmsd,mpnn,pTM,ipTM,multimer_pLDDT,num_interface_residues,num_hotspot_contacts,num_his_interface,num_his_total
0,1_2,1,DEVWERLLALLAAHPEASEVLALAEAARAHAEEARRALAAGDLAAA...,0.966230,52.795399,0.991804,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1_2,6,DEAWARLRALLAARPEAAEALALAEAAIAARAAARAALAAGDLATA...,0.945308,54.663250,0.974655,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
import os
for f in sorted(os.listdir('/content/params')):
    path = f'/content/params/{f}'
    print(f'{f}: {os.path.getsize(path) / 1024 / 1024:.2f} MB')

params_model_1_multimer_v3.npz: 0.00 MB
params_model_2_multimer_v3.npz: 0.00 MB
params_model_3_multimer_v3.npz: 0.00 MB
params_model_4_multimer_v3.npz: 0.00 MB
params_model_5_multimer_v3.npz: 0.00 MB
